# NHANES Cardiometabolic Exploration

Reads the star-schema marts in `exports/` (built by `scripts/build_dataset.py`) and charts the peer-group outlier detection: distributions, anomaly breakdowns, and where undiagnosed outliers cluster.

Run `python scripts/build_dataset.py` first if `exports/` is empty. This notebook only reads — it doesn't re-download or re-process anything.

In [ ]:
import warnings

# same pandas 2.2.2 / Python 3.14 false-positive as build_dataset.py
warnings.filterwarnings("ignore", category=FutureWarning, message=".*ChainedAssignmentError.*")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

EXPORT_DIR = Path("exports") if Path("exports").exists() else Path("../exports")

dim_respondents = pd.read_csv(EXPORT_DIR / "dim_respondents.csv")
fact_body = pd.read_csv(EXPORT_DIR / "fact_body_measures.csv")
fact_bp = pd.read_csv(EXPORT_DIR / "fact_blood_pressure.csv")
fact_labs = pd.read_csv(EXPORT_DIR / "fact_labs.csv")
fact_diagnoses = pd.read_csv(EXPORT_DIR / "fact_diagnoses.csv")
fact_anomalies = pd.read_csv(EXPORT_DIR / "fact_anomalies.csv")

# denormalized view for plotting — joins the star schema back together
wide = (
    dim_respondents
    .merge(fact_body, on="SEQN", how="left")
    .merge(fact_bp, on="SEQN", how="left")
    .merge(fact_labs, on="SEQN", how="left")
    .merge(fact_diagnoses, on="SEQN", how="left")
)

print(f"{len(wide):,} respondents, {len(fact_anomalies):,} flagged (respondent, field) pairs")

## Distributions by age band

BMI, blood pressure, and fasting glucose across the sample, so the peer-group z-score thresholds have visual context.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(
    axes,
    ["bmi", "mean_systolic", "fasting_glucose"],
    ["BMI", "Systolic BP (mmHg)", "Fasting Glucose (mg/dL)"],
):
    wide[col].dropna().hist(bins=40, ax=ax, color="steelblue")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Anomaly breakdown

Where the flagged (≥3 std dev from peer group) outliers concentrate — by field category and by age band.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

fact_anomalies["field_category"].value_counts().plot(
    kind="barh", ax=axes[0], color="crimson", title="Flagged outliers by category"
)
axes[0].invert_yaxis()

anomalies_with_age = fact_anomalies.merge(dim_respondents[["SEQN", "age_band"]], on="SEQN")
anomalies_with_age["age_band"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color="darkorange", title="Flagged outliers by age band"
)

plt.tight_layout()
plt.show()

## Undiagnosed outliers

BMI vs. fasting glucose, colored by self-reported diabetes diagnosis — outliers with no diagnosis on record are the interesting cluster.

In [ ]:
plot_df = wide.dropna(subset=["bmi", "fasting_glucose"])
colors = plot_df["diabetes_diagnosis"].map({"Yes": "crimson", "No": "steelblue", "Borderline": "orange"}).fillna("gray")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df["bmi"], plot_df["fasting_glucose"], c=colors, alpha=0.4, s=12)
ax.set_xlabel("BMI")
ax.set_ylabel("Fasting Glucose (mg/dL)")
ax.set_title("BMI vs. fasting glucose, colored by diagnosed diabetes status")

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=l)
           for l, c in [("Yes", "crimson"), ("No", "steelblue"), ("Borderline", "orange"), ("Not asked/unknown", "gray")]]
ax.legend(handles=handles, title="Diabetes diagnosis")
plt.show()

## Summary table

In [ ]:
summary = wide.groupby(["age_band", "gender"], observed=True).agg(
    respondents=("SEQN", "count"),
    avg_bmi=("bmi", "mean"),
    avg_systolic=("mean_systolic", "mean"),
    avg_glucose=("fasting_glucose", "mean"),
).round(1)
summary

## Regression: what predicts fasting glucose?

OLS regression of fasting glucose on BMI, waist circumference, systolic BP, and age. This is a second, independent way to spot outliers — the residual (actual minus predicted) flags respondents whose glucose is unusual given their other measurements, complementing the peer-group z-scores above.

In [ ]:
import numpy as np
import statsmodels.api as sm

reg_df = wide.dropna(subset=["fasting_glucose", "bmi", "waist_cm", "mean_systolic", "age_years"]).copy()

X = sm.add_constant(reg_df[["bmi", "waist_cm", "mean_systolic", "age_years"]])
y = reg_df["fasting_glucose"]
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
reg_df["predicted_glucose"] = model.predict(X)
reg_df["residual"] = reg_df["fasting_glucose"] - reg_df["predicted_glucose"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(reg_df["predicted_glucose"], reg_df["residual"], alpha=0.3, s=12, color="steelblue")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xlabel("Predicted fasting glucose")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs. predicted")

reg_df["residual"].hist(bins=40, ax=axes[1], color="steelblue")
axes[1].set_title("Residual distribution")

plt.tight_layout()
plt.show()

large_residual = reg_df[reg_df["residual"].abs() >= 2 * reg_df["residual"].std()]
print(f"{len(large_residual):,} respondents ({len(large_residual) / len(reg_df):.1%}) have a glucose "
      f"residual beyond 2 std devs of the model's prediction \u2014 glucose notably higher or lower "
      f"than their BMI/BP/age alone would predict.")

## Clustering: cardiometabolic risk phenotypes

K-means on the fasting-subsample labs (only respondents with a full lab panel — cholesterol, glucose, HbA1c, insulin, plus BMI/waist/BP) to see if the data separates into distinct risk groups *without* using the diagnosis labels at all. Diabetes diagnosis rate per cluster, computed after the fact, is the sanity check that the clusters mean something clinically.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

cluster_fields = [
    "bmi", "waist_cm", "mean_systolic", "mean_diastolic",
    "total_cholesterol", "hdl_cholesterol", "triglycerides", "ldl_cholesterol",
    "fasting_glucose", "hba1c", "insulin",
]
cluster_df = wide.dropna(subset=cluster_fields).copy()

X_scaled = StandardScaler().fit_transform(cluster_df[cluster_fields])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_df["cluster"] = kmeans.fit_predict(X_scaled)

print(f"{len(cluster_df):,} respondents with a full lab panel, clustered into {kmeans.n_clusters} groups")
cluster_df["cluster"].value_counts().sort_index()

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=cluster_df["cluster"], cmap="viridis", alpha=0.5, s=12)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Cardiometabolic clusters (PCA projection)")
plt.colorbar(scatter, label="Cluster")
plt.show()

In [ ]:
profile = cluster_df.groupby("cluster")[cluster_fields].mean().round(1)
profile["respondents"] = cluster_df.groupby("cluster").size()
profile["diabetes_rate"] = cluster_df.groupby("cluster")["diabetes_diagnosis"].apply(
    lambda s: (s == "Yes").mean()
).round(2)
profile

## Lean Six Sigma: process capability against clinical reference ranges

Treating each lab/vital as a "process" and its established clinical normal range as spec limits (LSL/USL), the standard Six Sigma capability metrics apply directly: **Cp** (potential capability, ignores centering), **Cpk** (actual capability, accounts for how centered the population is within spec), **% out of spec**, **DPMO**, and the equivalent **sigma level** (long-term convention with the standard 1.5-sigma shift). A lower Cpk/sigma level means more of the sampled population falls outside the healthy range — either real prevalence of a condition or population variability, not a measurement defect, but the same math applies.

In [ ]:
from scipy.stats import norm

# (LSL, USL, unit) per metric \u2014 established clinical reference ranges.
# LSL is None where no clinically meaningful lower bound is commonly used.
SPEC_LIMITS = {
    "bmi":              (18.5, 25.0, "kg/m^2"),
    "mean_systolic":    (90.0, 120.0, "mmHg"),
    "fasting_glucose":  (70.0, 99.0, "mg/dL"),
    "hba1c":            (None, 5.7, "%"),
    "total_cholesterol": (None, 200.0, "mg/dL"),
    "ldl_cholesterol":  (None, 100.0, "mg/dL"),
}


def capability(series, lsl, usl):
    s = series.dropna()
    mean, std = s.mean(), s.std()
    cp = (usl - lsl) / (6 * std) if lsl is not None else np.nan
    cpu = (usl - mean) / (3 * std)
    cpl = (mean - lsl) / (3 * std) if lsl is not None else np.inf
    cpk = min(cpu, cpl)
    frac_out = ((s < (lsl if lsl is not None else -np.inf)) | (s > usl)).mean()
    dpmo = frac_out * 1_000_000
    sigma_level = norm.isf(frac_out) + 1.5 if 0 < frac_out < 1 else np.nan
    return pd.Series({
        "n": len(s), "mean": round(mean, 1), "std": round(std, 2),
        "Cp": round(cp, 2) if not np.isnan(cp) else None,
        "Cpk": round(cpk, 2),
        "pct_out_of_spec": round(frac_out * 100, 1),
        "DPMO": round(dpmo),
        "sigma_level": round(sigma_level, 2) if not np.isnan(sigma_level) else None,
    })


rows = {}
for field, (lsl, usl, unit) in SPEC_LIMITS.items():
    rows[f"{field} ({unit})"] = capability(wide[field], lsl, usl)

capability_table = pd.DataFrame(rows).T
capability_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
capability_table["Cpk"].astype(float).plot(kind="barh", ax=ax, color="teal")
ax.axvline(1.33, color="black", linestyle="--", linewidth=1, label="Cpk = 1.33 (common minimum target)")
ax.set_xlabel("Cpk")
ax.set_title("Process capability (Cpk) by metric")
ax.legend()
plt.tight_layout()
plt.show()

## Regression target 2: blood pressure by diagnosed condition

Same idea as the glucose regression above, but now the diagnosed conditions (diabetes, high cholesterol, smoking status) go in as categorical **factors** alongside BMI/waist/age, rather than as a target — this quantifies how much each condition shifts systolic BP *after* controlling for body measures and age, i.e. comparing condition groups on equal footing rather than raw group averages.

In [ ]:
import statsmodels.formula.api as smf

cond_df = wide.dropna(subset=[
    "mean_systolic", "bmi", "waist_cm", "age_years",
    "diabetes_diagnosis", "high_cholesterol_diagnosis", "smoking_status",
]).copy()
# drop the small Borderline diabetes group so the factor is a clean Yes/No comparison
cond_df = cond_df[cond_df["diabetes_diagnosis"] != "Borderline"]

condition_model = smf.ols(
    "mean_systolic ~ bmi + waist_cm + age_years "
    "+ C(diabetes_diagnosis, Treatment(reference='No')) "
    "+ C(high_cholesterol_diagnosis, Treatment(reference='No')) "
    "+ C(smoking_status, Treatment(reference='Never smoker'))",
    data=cond_df,
).fit()
print(condition_model.summary())

In [ ]:
# isolate just the condition-factor coefficients (the C(...) terms)
condition_terms = [t for t in condition_model.params.index if t.startswith("C(")]
condition_effects = pd.DataFrame({
    "systolic_mmHg_effect": condition_model.params[condition_terms].round(2),
    "p_value": condition_model.pvalues[condition_terms].round(4),
}).sort_values("systolic_mmHg_effect", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["crimson" if p < 0.05 else "lightgray" for p in condition_effects["p_value"]]
ax.barh(condition_effects.index, condition_effects["systolic_mmHg_effect"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Effect on mean systolic BP (mmHg), vs. reference group")
ax.set_title("Condition effects on blood pressure, controlling for BMI/waist/age\n(red = p < 0.05)")
plt.tight_layout()
plt.show()
condition_effects

## Regression target 3: predicting diabetes diagnosis from biomarkers

Flips the setup around — diagnosis is now the **target** (logistic regression, Yes/No) instead of a factor, predicted from BMI, waist, systolic BP, age, total cholesterol, and fasting glucose. Coefficients are reported as odds ratios: an odds ratio of 1.5 for a predictor means each one-unit increase multiplies the odds of a diabetes diagnosis by 1.5x, holding the others constant.

In [ ]:
logit_fields = ["bmi", "waist_cm", "mean_systolic", "age_years", "total_cholesterol", "fasting_glucose"]

logit_df = wide[wide["diabetes_diagnosis"].isin(["Yes", "No"])].dropna(subset=logit_fields).copy()
logit_df["diabetes_binary"] = (logit_df["diabetes_diagnosis"] == "Yes").astype(int)

X_logit = sm.add_constant(logit_df[logit_fields])
y_logit = logit_df["diabetes_binary"]
logit_model = sm.Logit(y_logit, X_logit).fit()
print(logit_model.summary())

odds_ratios = np.exp(logit_model.params).rename("odds_ratio").round(3)
print("\nOdds ratios:")
print(odds_ratios)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

predicted_prob = logit_model.predict(X_logit)
auc = roc_auc_score(y_logit, predicted_prob)
fpr, tpr, _ = roc_curve(y_logit, predicted_prob)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color="steelblue", label=f"AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Predicting diabetes diagnosis from biomarkers alone")
ax.legend()
plt.show()

print(f"{y_logit.sum():,} of {len(y_logit):,} respondents ({y_logit.mean():.1%}) have a diabetes "
      f"diagnosis in this fully-observed biomarker subset.")